# Data Cleaning

O MovieLens é relativamente limpo, mas ainda tem problemas reais que precisam ser tratados antes de qualquer cálculo.

## 1. Filmes

In [18]:
import pandas as pd
import re

movies = pd.read_csv("../data/raw/ml-small/movies.csv")
print(movies.head(5).to_string(index=False))

 movieId                              title                                      genres
       1                   Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy
       2                     Jumanji (1995)                  Adventure|Children|Fantasy
       3            Grumpier Old Men (1995)                              Comedy|Romance
       4           Waiting to Exhale (1995)                        Comedy|Drama|Romance
       5 Father of the Bride Part II (1995)                                      Comedy


### 1.1 Título

O título dos filmes vem com o ano embutido — "Pulp Fiction (1994)". Se usar o título como texto para TF-IDF, o modelo vai tratar (1994) como um token relevante e criar ruído.

**O ano precisa ser extraído como campo separado e removido do título.**

In [10]:
def extrair_ano_do_titulo(titulo: str) -> tuple[str, int | None]:
    """
    Extrai o ano do título e retorna (titulo_limpo, ano).
 
    Casos tratados:
      'Toy Story (1995)'                               - ('Toy Story', 1995)
      'Usual Suspects, The (1995)'                     - ('The Usual Suspects', 1995)
      'Dumb & Dumber (Dumb and Dumber) (1994)'         - ('Dumb & Dumber', 1994)
      'Filme Sem Ano'                                  - ('Filme Sem Ano', None)
    """
    # Captura o último (YYYY) — evita capturar anos no meio como '2001: A Space Odyssey'
    match_ano = re.search(r"\((\d{4})\)\s*$", titulo)
    ano = int(match_ano.group(1)) if match_ano else None
 
    # Remove o ano do título
    titulo_limpo = re.sub(r"\((\d{4})\)\s*$", "", titulo).strip()
 
    # Remove subtítulos alternativos longos entre parênteses que o MovieLens inclui
    # Ex: 'Dumb & Dumber (Dumb and Dumber)' → 'Dumb & Dumber'
    titulo_limpo = re.sub(r"\([^)]{10,}\)\s*$", "", titulo_limpo).strip()
 
    # Corrige artigos invertidos pelo MovieLens para ordenação alfabética
    # Ex: 'Usual Suspects, The' → 'The Usual Suspects'
    match_artigo = re.match(
        r"^(.*),\s*(The|A|An|Les|Le|La|Los|Las|El|Das|Die|Der)\s*$",
        titulo_limpo,
        re.IGNORECASE
    )
    if match_artigo:
        titulo_limpo = f"{match_artigo.group(2)} {match_artigo.group(1)}"
 
    return titulo_limpo, ano

### 1.2 Gêneros

Os gêneros vêm como uma string única separada por pipe — "Crime|Thriller|Comedy". Para o modelo, isso é uma palavra só. Você precisa separar em lista para poder fazer multi-hot encoding, onde cada gênero vira uma coluna binária independente.

Alguns filmes têm gênero "(no genres listed)". Esses registros não podem entrar na filtragem de conteúdo baseada em gênero. Existem duas opções: remover esses filmes do sistema ou tratar como categoria especial.

**Decisão final:**
- tratar como categoria especial [NO_GENRE_PLACEHOLDER]

In [11]:
def processar_generos(generos_str: str) -> list[str]:
    """  
    Alternativas consideradas e descartadas:
      - Remover o filme: perde dados colaborativos valiosos (ratings existem)
      - Deixar lista vazia: quebra o multi-hot encoding downstream
      - Marcar como NaN: propaga problemas em operações de agrupamento
    """
    if pd.isna(generos_str) or generos_str.strip() == "(no genres listed)":
        return [NO_GENRE_PLACEHOLDER]
 
    return [g.strip() for g in generos_str.split("|") if g.strip()]

## Orquestração && Execução

In [13]:
NO_GENRE_PLACEHOLDER = "Unknown"

def limpar_movies(caminho_entrada: str, caminho_saida: str) -> pd.DataFrame:
    """
    Operações:
      1. Extração do ano e limpeza do título
      2. Conversão de gêneros para lista
      3. Remoção de filmes com título duplicado (mantém o de menor movieId)
      4. Garantia de que todo filme tem ao menos um gênero
      5. Exportação do CSV limpo
 
    Retorna o DataFrame limpo com genres ainda como lista (para uso imediato).
    O CSV exportado serializa a lista como 'A|B|C' para compatibilidade.
    """
    df = pd.read_csv(caminho_entrada)
 
    # ── 1. Título e ano ───────────────────────────────────────────────────────
    resultado = df["title"].apply(extrair_ano_do_titulo)
    df["title"] = resultado.apply(lambda x: x[0])
    df["year"]  = resultado.apply(lambda x: x[1])
 
    # ── 2. Gêneros ────────────────────────────────────────────────────────────
    df["genres"] = df["genres"].apply(processar_generos)
 
    # ── 3. Remove duplicatas de título (mantém menor movieId — mais canônico) ─
    df = df.sort_values("movieId")
    df = df.drop_duplicates(subset="title", keep="first").reset_index(drop=True)
 
    # ── 4. Validação: todo filme deve ter ao menos um gênero ──────────────────
    sem_genero = df["genres"].apply(len) == 0
    assert not sem_genero.any(), (
        f"{sem_genero.sum()} filmes ficaram sem gênero após o processamento. "
        f"IDs: {df[sem_genero]['movieId'].tolist()}"
    )
 
    # ── 5. Exportação ─────────────────────────────────────────────────────────
    df_saida = df[["movieId", "title", "year", "genres"]].copy()
    df_saida["genres"] = df_saida["genres"].apply("|".join)
    df_saida.to_csv(caminho_saida, index=False)
 
    return df

if __name__ == "__main__":
    df = limpar_movies("../data/raw/ml-small/movies.csv", "../data/processed/movies_clean.csv")
    print(df[["movieId", "title", "year", "genres"]].head(10).to_string(index=False))

 movieId                       title   year                                            genres
       1                   Toy Story 1995.0 [Adventure, Animation, Children, Comedy, Fantasy]
       2                     Jumanji 1995.0                    [Adventure, Children, Fantasy]
       3            Grumpier Old Men 1995.0                                 [Comedy, Romance]
       4           Waiting to Exhale 1995.0                          [Comedy, Drama, Romance]
       5 Father of the Bride Part II 1995.0                                          [Comedy]
       6                        Heat 1995.0                         [Action, Crime, Thriller]
       7                     Sabrina 1995.0                                 [Comedy, Romance]
       8                Tom and Huck 1995.0                             [Adventure, Children]
       9                Sudden Death 1995.0                                          [Action]
      10                   GoldenEye 1995.0                 

## Ratings

Ratings duplicados podem existir quando um usuário avalia o mesmo filme mais de uma vez. 

Usuários com poucos ratings (menos de 5, por exemplo) são problemáticos para o colaborativo. A similaridade calculada com base em 2 ou 3 filmes é estatisticamente fraca.

Filmes com poucos ratings também criam ruído — um filme com 2 avaliações de 5 estrelas não é necessariamente melhor que Pulp Fiction. 

### 2.1 Duplicatas

Fica com o rating mais recente, pois o gosto do usuário evoluiu.

In [30]:
MIN_RATINGS_USUARIO = 15   # usuários com menos ratings que isso são removidos
MIN_RATINGS_FILME   = 10   # filmes com menos ratings que isso são removidos

def remover_duplicatas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove ratings duplicados — mesmo userId + movieId avaliado mais de uma vez.
    Mantém apenas o rating mais recente (maior timestamp), pois o gosto do
    usuário evolui e a avaliação mais nova é a mais representativa.
    """
    antes = len(df)
 
    df = (
        df.sort_values("timestamp", ascending=False)
          .drop_duplicates(subset=["userId", "movieId"], keep="first")
          .sort_values(["userId", "movieId"])
          .reset_index(drop=True)
    )
 
    removidos = antes - len(df)
    if removidos:
        print(f"[duplicatas] {removidos} ratings duplicados removidos "
              f"(mantido o mais recente por userId+movieId).")
    else:
        print("[duplicatas] Nenhuma duplicata encontrada.")
 
    return df

### 2.2 Usuários com poucos ratings

Define um threshold mínimo de 5 ratings e filtra esses usuários do treino.

In [20]:
def filtrar_usuarios(df: pd.DataFrame, min_ratings: int = MIN_RATINGS_USUARIO) -> pd.DataFrame:
    """
    Remove usuários com menos de `min_ratings` avaliações.
    Usuários com histórico muito curto geram vetores instáveis — a similaridade
    calculada com 2 ou 3 filmes não é estatisticamente confiável para o colaborativo.
    """
    contagem = df.groupby("userId")["movieId"].count()
    usuarios_validos = contagem[contagem >= min_ratings].index
 
    antes = df["userId"].nunique()
    df = df[df["userId"].isin(usuarios_validos)].reset_index(drop=True)
    removidos = antes - df["userId"].nunique()
 
    print(f"[usuários]  threshold={min_ratings} | "
          f"{removidos} usuários removidos | "
          f"{df['userId'].nunique()} restantes.")
 
    return df

### 2.3 Filmes com poucos ratings

Filtra pelo mínimo de 10 ratings antes de qualquer cálculo de popularidade.

In [21]:
def filtrar_filmes(df: pd.DataFrame, min_ratings: int = MIN_RATINGS_FILME) -> pd.DataFrame:
    """
    Remove filmes com menos de `min_ratings` avaliações.
    Filmes com poucos ratings têm médias infladas ou deprimidas por acaso —
    um filme com 2 notas 5.0 não é necessariamente melhor que Pulp Fiction.
    Também geram vetores latentes instáveis no SVD.
    """
    contagem = df.groupby("movieId")["userId"].count()
    filmes_validos = contagem[contagem >= min_ratings].index
 
    antes = df["movieId"].nunique()
    df = df[df["movieId"].isin(filmes_validos)].reset_index(drop=True)
    removidos = antes - df["movieId"].nunique()
 
    print(f"[filmes]    threshold={min_ratings} | "
          f"{removidos} filmes removidos | "
          f"{df['movieId'].nunique()} restantes.")
 
    return df

## Orquestração && Execução

In [31]:
def limpar_ratings(
    caminho_entrada: str,
    caminho_saida: str,
    min_ratings_usuario: int = MIN_RATINGS_USUARIO,
    min_ratings_filme: int = MIN_RATINGS_FILME
) -> pd.DataFrame:
    """
    Pipeline completo de limpeza do ratings.csv do MovieLens.
 
    Ordem das operações é intencional:
      1. Duplicatas primeiro — garante que a contagem de ratings por usuário/filme
         nos passos seguintes reflita avaliações únicas, não repetições.
      2. Usuários depois — remove ruído humano antes de avaliar cobertura dos filmes.
      3. Filmes por último — após remover usuários ruidosos, alguns filmes podem
         cair abaixo do threshold e também precisam sair.
 
    Retorna o DataFrame limpo.
    """
    df = pd.read_csv(caminho_entrada)
 
    # print(f"[entrada]   {len(df)} ratings | "
    #       f"{df['userId'].nunique()} usuários | "
    #       f"{df['movieId'].nunique()} filmes\n")
 
    df = remover_duplicatas(df)
    df = filtrar_usuarios(df, min_ratings=min_ratings_usuario)
    df = filtrar_filmes(df, min_ratings=min_ratings_filme)
 
    df.to_csv(caminho_saida, index=False)
 
    # print(f"\n[saída]     {len(df)} ratings | "
    #       f"{df['userId'].nunique()} usuários | "
    #       f"{df['movieId'].nunique()} filmes")
    # print(f"[arquivo]   salvo em '{caminho_saida}'")
 
    return df
 
if __name__ == "__main__":
    limpar_ratings("../data/raw/ml-small/ratings.csv", "../data/processed/ratings_clean.csv")

[duplicatas] Nenhuma duplicata encontrada.
[usuários]  threshold=15 | 0 usuários removidos | 610 restantes.
[filmes]    threshold=10 | 7455 filmes removidos | 2269 restantes.
